In [32]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jeromeblanchet/commonsenseqa-nlp-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/jeromeblanchet/commonsenseqa-nlp-dataset


In [33]:
import os
print("Dataset path:", path)

print("\nFiles:")
for file in os.listdir(path):
    print(file)

Dataset path: /kaggle/input/datasets/jeromeblanchet/commonsenseqa-nlp-dataset

Files:
dev_rand_split.jsonl
dev_rand_split_EASY.jsonl
test_rand_split_no_answers.jsonl
train_rand_split_EASY.jsonl
train_rand_split.jsonl


In [34]:
from pprint import pprint
import json
import os

train_file = os.path.join(path, "train_rand_split.jsonl")

with open(train_file, "r", encoding="utf-8") as f:
    sample = json.loads(next(f))

pprint(sample)

{'answerKey': 'A',
 'id': '075e483d21c29a511267ef62bedc0461',
 'question': {'choices': [{'label': 'A', 'text': 'ignore'},
                          {'label': 'B', 'text': 'enforce'},
                          {'label': 'C', 'text': 'authoritarian'},
                          {'label': 'D', 'text': 'yell at'},
                          {'label': 'E', 'text': 'avoid'}],
              'question_concept': 'punishing',
              'stem': 'The sanctions against the school were a punishing blow, '
                      'and they seemed to what the efforts the school had made '
                      'to change?'}}


In [35]:
import json
import os

train_file = os.path.join(path, "train_rand_split.jsonl")

with open(train_file, "r", encoding="utf-8") as f:

    for i, line in enumerate(f):

        sample = json.loads(line)

        question = sample["question"]["stem"]

        answer_key = sample["answerKey"]

        # Convert choices into a dictionary
        choices = {
            choice["label"]: choice["text"]
            for choice in sample["question"]["choices"]
        }

        answer = choices[answer_key]

        print("Question :", question)
        print("Answer   :", answer)
        print("-" * 80)

        if i == 4:   # Show first 5 samples
            break

Question : The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?
Answer   : ignore
--------------------------------------------------------------------------------
Question : Sammy wanted to go to where the people were.  Where might he go?
Answer   : populated areas
--------------------------------------------------------------------------------
Question : To locate a choker not located in a jewelry box or boutique where would you go?
Answer   : jewelry store
--------------------------------------------------------------------------------
Question : Google Maps and other highway and street GPS services have replaced what?
Answer   : atlas
--------------------------------------------------------------------------------
Question : The fox walked from the city into the forest, what was it looking for?
Answer   : natural habitat
--------------------------------------------------------------------------------


In [36]:
PROMPT_TEMPLATES = [
    "{}",
    "Answer the following question: {}",
    "Provide the best answer: {}",
    "What is the correct answer? {}",
    "Can you answer this? {}"
]

In [37]:
def format_answer(answer):
    answer = answer.strip()

    if not answer.endswith("."):
        answer += "."

    return answer[0].upper() + answer[1:]

In [38]:
RESPONSE_TEMPLATES = [
    "{}.",
    "The answer is {}.",
    "The correct answer is {}.",
    "It is {}.",
    "{} is the correct answer.",
    "Usually {}.",
    "Most commonly {}.",
    "Typically {}.",
    "The best answer is {}.",
    "The most appropriate answer is {}.",
    "The correct choice is {}.",
    "The right answer is {}.",
    "People generally choose {}.",
    "A suitable answer is {}.",
    "The expected answer is {}."
]

In [39]:
import json
import os
import random
import pandas as pd

TARGET_SAMPLES = 15000

def format_answer(answer):
    answer = answer.strip()

    if answer:
        answer = answer[0].upper() + answer[1:]

    template = random.choice(RESPONSE_TEMPLATES)
    response = template.format(answer)

    if not response.endswith("."):
        response += "."

    return response


files = [
    os.path.join(path, "train_rand_split.jsonl"),
    os.path.join(path, "dev_rand_split.jsonl")
]

dataset = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            sample = json.loads(line)

            question = sample["question"]["stem"].strip()

            choices = {
                c["label"]: c["text"]
                for c in sample["question"]["choices"]
            }

            answer = choices[sample["answerKey"]]

            for prompt_template in PROMPT_TEMPLATES:
                prompt = prompt_template.format(question)
                response = format_answer(answer)

                dataset.append({
                    "prompt": prompt,
                    "response": response
                })

print(f"Generated samples: {len(dataset):,}")

random.shuffle(dataset)

if len(dataset) > TARGET_SAMPLES:
    dataset = dataset[:TARGET_SAMPLES]

print(f"Saving samples: {len(dataset):,}")

df = pd.DataFrame(dataset)

df.to_csv("reasoning.csv", index=False, encoding="utf-8")

print(df.head())

Generated samples: 54,810
Saving samples: 15,000
                                              prompt  \
0  Can you answer this? What is helping likely to...   
1  What is the correct answer? Billy set of the c...   
2  Answer the following question: You can find ca...   
3  The man took paperwork to other people to cons...   
4  Food on what kind of transport is normally fre...   

                             response  
0  The right answer is Good feelings.  
1         A suitable answer is Floor.  
2                     Usually Casino.  
3     The expected answer is Meeting.  
4        Most commonly Space shuttle.  


In [40]:
!pip -q install datasets transformers sentencepiece tqdm

In [41]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "open-thoughts/OpenThoughts-114k",
    split="train"
)

print(dataset)
print(dataset[0])

Dataset({
    features: ['system', 'conversations'],
    num_rows: 113957
})
{'system': "Your role as an assistant involves thoroughly exploring questions through a systematic long thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracing, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution. In the Thought section, detail your reasoning process using the specified format: <|begin_of_thought|> {thought with steps separated with '\\n\\n'} <|end_of_thought|> Each step should include detailed considerations such as analisying questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous steps. In the Solution section, based on various attempts, explorations, and reflections 

In [42]:
print(dataset.column_names)

for column in dataset.column_names:
    print(f"\n{column}")
    print(dataset[column][0])

['system', 'conversations']

system
Your role as an assistant involves thoroughly exploring questions through a systematic long thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracing, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution. In the Thought section, detail your reasoning process using the specified format: <|begin_of_thought|> {thought with steps separated with '\n\n'} <|end_of_thought|> Each step should include detailed considerations such as analisying questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous steps. In the Solution section, based on various attempts, explorations, and reflections from the Thought section, systematically present the fi

In [43]:
sample = dataset[0]

for key, value in sample.items():
    print(f"\n{'='*20} {key} {'='*20}")
    print(value)


==================== system ====================
Your role as an assistant involves thoroughly exploring questions through a systematic long thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracing, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution. In the Thought section, detail your reasoning process using the specified format: <|begin_of_thought|> {thought with steps separated with '\n\n'} <|end_of_thought|> Each step should include detailed considerations such as analisying questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous steps. In the Solution section, based on various attempts, explorations, and reflections from the Thought section, systematically 

In [44]:
pairs = []

for sample in dataset:
    conv = sample["conversations"]

    if len(conv) >= 2:
        if conv[0]["from"] == "user" and conv[1]["from"] == "assistant":
            pairs.append({
                "prompt": conv[0]["value"].strip(),
                "response": conv[1]["value"].strip()
            })

print(len(pairs))
print(pairs[0])

113957
{'prompt': 'Generate an executable Python function generated from the given prompt. The function should take stdin as input and print the output. Simply call the function after the definition.The Chef likes to stay in touch with his staff. So, the Chef, the head server, and the sous-chef all carry two-way transceivers so they can stay in constant contact. Of course, these transceivers have a limited range so if two are too far apart, they cannot communicate directly.\n\n\nThe Chef invested in top-of-the-line transceivers which have a few advanced features. One is that even if two people cannot talk directly because they are out of range, if there is another transceiver that is close enough to both, then the two transceivers can still communicate with each other using the third transceiver as an intermediate device.\n\n\nThere has been a minor emergency in the Chef\'s restaurant\nand he needs to communicate with both the head server and the sous-chef right away. Help the Chef det

In [45]:
pairs = []

for sample in dataset:
    conv = sample["conversations"]

    if len(conv) != 2:
        continue

    if conv[0]["from"] != "user" or conv[1]["from"] != "assistant":
        continue

    prompt = conv[0]["value"].strip()
    response = conv[1]["value"].strip()

    if len(prompt) == 0 or len(response) == 0:
        continue

    if len(response) > 5000:
        continue

    pairs.append({
        "prompt": prompt,
        "response": response
    })

print("Total pairs:", len(pairs))
print()
print("Prompt:\n", pairs[0]["prompt"][:500])
print("\n" + "-"*80 + "\n")
print("Response:\n", pairs[0]["response"][:1000])

Total pairs: 8108

Prompt:
 Generate an executable Python function generated from the given prompt. The function should take stdin as input and print the output. Simply call the function after the definition.Having learned the multiplication table, Takahashi can multiply two integers between 1 and 9 (inclusive) together. He cannot do any other calculation.

Given are two integers A and B.

If Takahashi can calculate A \times B, print the result; if he cannot, print `-1` instead.

Constraints

* 1 \leq A \leq 20
* 1 \leq B 

--------------------------------------------------------------------------------

Response:
 <|begin_of_thought|>

Okay, let's see. The problem says that Takahashi can multiply two integers between 1 and 9 inclusive. So if either A or B is outside that range, he can't calculate the product, right? So I need to check if both A and B are in 1 to 9. If yes, then output their product. Otherwise, output -1.

Wait, but the input constraints say that A and B can be up to 2

In [46]:
import re

for pair in pairs:
    pair["response"] = re.sub(
        r"<\|begin_of_thought\|>|<\|end_of_thought\|>|<\|begin_of_solution\|>|<\|end_of_solution\|>",
        "",
        pair["response"],
    ).strip()

print(pairs[0]["response"][:1500])

Okay, let's see. The problem says that Takahashi can multiply two integers between 1 and 9 inclusive. So if either A or B is outside that range, he can't calculate the product, right? So I need to check if both A and B are in 1 to 9. If yes, then output their product. Otherwise, output -1.

Wait, but the input constraints say that A and B can be up to 20. So the input could be, for example, 10 and 5. Since 10 is not between 1 and 9, even though B is 5, which is acceptable, the whole product can't be calculated. So the condition is that both A and B must be between 1 and 9 inclusive. Otherwise, output -1.

So the steps are:

1. Read A and B from input.
2. Check if both A and B are in the range 1-9.
3. If yes, compute A*B and print it.
4. If no, print -1.

Let me look at the examples to confirm.

First example: 2 and 5. Both are between 1-9. So output 10. Correct.

Second example: 5 and 10. 10 is outside. So output -1. Correct.

Third example: 9 and 9. Both are in range. So output 81. Co

In [48]:
import os
import pandas as pd
from tokenizers import Tokenizer

MAX_TOKENS = 1024

tokenizer = Tokenizer.from_file(
    "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"
)

filtered_pairs = []

for pair in pairs:
    prompt = str(pair["prompt"]).strip()
    response = str(pair["response"]).strip()

    total_tokens = len(tokenizer.encode(prompt + "\n" + response).ids)

    if total_tokens <= MAX_TOKENS:
        filtered_pairs.append({
            "prompt": prompt,
            "response": response
        })

print(f"Kept: {len(filtered_pairs):,} / {len(pairs):,}")

new_df = pd.DataFrame(filtered_pairs)

if os.path.exists("reasoning.csv"):
    existing_df = pd.read_csv("reasoning.csv")
    combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    combined_df.drop_duplicates(subset=["prompt", "response"], inplace=True)
else:
    combined_df = new_df

combined_df.to_csv("reasoning.csv", index=False, encoding="utf-8")

print(f"Added: {len(new_df):,}")
print(f"Total: {len(combined_df):,}")

Kept: 1,619 / 8,108
Added: 1,619
Total: 23,108


In [49]:
tokenizer = Tokenizer.from_file(
    "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"
)

MAX_TOKENS = 1024

In [52]:
dataset = load_dataset(
    "allenai/ai2_arc",
    "ARC-Challenge"
)

train = dataset["train"]

print(train)
print(f"Total training samples: {len(train):,}")

README.md: 0.00B [00:00, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…):   0%|          | 0.00/190k [00:00<?, ?B/s]

ARC-Challenge/test-00000-of-00001.parque(…):   0%|          | 0.00/204k [00:00<?, ?B/s]

ARC-Challenge/validation-00000-of-00001.(…):   0%|          | 0.00/55.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'choices', 'answerKey'],
    num_rows: 1119
})
Total training samples: 1,119


In [53]:
MAX_TOKENS = 1024

In [54]:
PROMPT_TEMPLATES = [
    "Answer the following multiple-choice question.\n\n{}",
    "Choose the correct answer.\n\n{}",
    "Solve this science reasoning question.\n\n{}",
    "Read the question and determine the correct answer.\n\n{}",
    "Find the best answer for the following question.\n\n{}",
    "Select the correct option.\n\n{}"
]

pairs = []

for sample in train:

    question = sample["question"]

    choices = sample["choices"]

    labels = choices["label"]
    texts = choices["text"]

    options = []
    answer = None

    for label, text in zip(labels, texts):
        options.append(f"{label}. {text}")

        if label == sample["answerKey"]:
            answer = text

    question_text = (
        question.strip()
        + "\n\n"
        + "\n".join(options)
    )

    prompt = random.choice(PROMPT_TEMPLATES).format(question_text)
    response = answer.strip()

    total_tokens = len(
        tokenizer.encode(prompt + "\n" + response).ids
    )

    if total_tokens <= MAX_TOKENS:
        pairs.append({
            "prompt": prompt,
            "response": response
        })

print(f"Generated pairs: {len(pairs):,}")
print(pairs[0])

Generated pairs: 1,119
{'prompt': 'Solve this science reasoning question.\n\nGeorge wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?\n\nA. dry palms\nB. wet palms\nC. palms covered with oil\nD. palms covered with lotion', 'response': 'dry palms'}


In [55]:
new_df = pd.DataFrame(pairs)

if os.path.exists("reasoning.csv"):
    existing_df = pd.read_csv("reasoning.csv")

    combined_df = pd.concat(
        [existing_df, new_df],
        ignore_index=True
    )

    combined_df.drop_duplicates(
        subset=["prompt", "response"],
        inplace=True
    )

else:
    combined_df = new_df

combined_df.to_csv(
    "reasoning.csv",
    index=False,
    encoding="utf-8"
)

print(f"Added: {len(new_df):,}")
print(f"Total: {len(combined_df):,}")


Added: 1,119
Total: 24,227


In [56]:
dataset = load_dataset(
    "allenai/winogrande",
    "winogrande_xl"
)

train = dataset["train"]

print(train)
print(f"Total training samples: {len(train):,}")

README.md: 0.00B [00:00, ?B/s]

winogrande_xl/train-00000-of-00001.parqu(…):   0%|          | 0.00/2.06M [00:00<?, ?B/s]

winogrande_xl/test-00000-of-00001.parque(…):   0%|          | 0.00/118k [00:00<?, ?B/s]

winogrande_xl/validation-00000-of-00001.(…):   0%|          | 0.00/85.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40398 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1767 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1267 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence', 'option1', 'option2', 'answer'],
    num_rows: 40398
})
Total training samples: 40,398


In [57]:
PROMPT_TEMPLATES = [
    "Complete the sentence with the most appropriate choice.\n\n{}",
    "Choose the correct word to complete the sentence.\n\n{}",
    "Select the best option for the blank.\n\n{}",
    "Fill in the blank using commonsense reasoning.\n\n{}",
    "Determine the correct completion.\n\n{}",
    "Read the sentence carefully and choose the correct answer.\n\n{}",
    "Which option best completes the sentence?\n\n{}"
]

pairs = []

for sample in train:

    sentence = sample["sentence"].strip()
    option1 = sample["option1"].strip()
    option2 = sample["option2"].strip()

    question = (
        f"{sentence}\n\n"
        f"Option 1: {option1}\n"
        f"Option 2: {option2}"
    )

    prompt = random.choice(PROMPT_TEMPLATES).format(question)

    if sample["answer"] == "1":
        response = option1
    else:
        response = option2

    total_tokens = len(
        tokenizer.encode(prompt + "\n" + response).ids
    )

    if total_tokens <= MAX_TOKENS:
        pairs.append({
            "prompt": prompt,
            "response": response
        })

print(f"Generated pairs: {len(pairs):,}")
print()
print(pairs[0])

Generated pairs: 40,398

{'prompt': "Complete the sentence with the most appropriate choice.\n\nIan volunteered to eat Dennis's menudo after already having a bowl because _ despised eating intestine.\n\nOption 1: Ian\nOption 2: Dennis", 'response': 'Dennis'}


In [62]:
PROMPT_TEMPLATES = [
    "Complete the sentence with the most appropriate choice.\n\n{}",
    "Choose the correct word to complete the sentence.\n\n{}",
    "Select the best option for the blank.\n\n{}",
    "Fill in the blank using commonsense reasoning.\n\n{}",
    "Determine the correct completion.\n\n{}",
    "Read the sentence carefully and choose the correct answer.\n\n{}",
    "Which option best completes the sentence?\n\n{}"
]

RESPONSE_TEMPLATES = [
    "The correct answer is **{}** because it best fits the context.",
    "{} is the correct choice because it completes the sentence logically.",
    "Based on the sentence, the correct answer is {}.",
    "{} best completes the sentence according to the context.",
    "The most appropriate answer is {} because it matches the sentence.",
    "The answer is {} since it is the only option consistent with the context.",
    "{} is correct because it makes the sentence meaningful and coherent."
]

pairs = []

for sample in train:

    sentence = sample["sentence"].strip()
    option1 = sample["option1"].strip()
    option2 = sample["option2"].strip()

    question = (
        f"{sentence}\n\n"
        f"Option 1: {option1}\n"
        f"Option 2: {option2}"
    )

    prompt = random.choice(PROMPT_TEMPLATES).format(question)

    if sample["answer"] == "1":
        correct = option1
    else:
        correct = option2

    response = random.choice(RESPONSE_TEMPLATES).format(correct)

    total_tokens = len(
        tokenizer.encode(prompt + "\n" + response).ids
    )

    if total_tokens <= MAX_TOKENS:
        pairs.append({
            "prompt": prompt,
            "response": response
        })

print(f"Generated pairs: {len(pairs):,}")
print()
print(pairs[0])

Generated pairs: 40,398

{'prompt': "Read the sentence carefully and choose the correct answer.\n\nIan volunteered to eat Dennis's menudo after already having a bowl because _ despised eating intestine.\n\nOption 1: Ian\nOption 2: Dennis", 'response': 'Dennis best completes the sentence according to the context.'}


In [68]:
import pprint
pprint.pp(pairs[10])

{'prompt': 'Choose the correct word to complete the sentence.\n'
           '\n'
           'The treasury workers took the gold bars off of the trolley and '
           'stacked them in the safe until the _ was empty.\n'
           '\n'
           'Option 1: safe\n'
           'Option 2: trolley',
 'response': 'trolley is correct because it makes the sentence meaningful and '
             'coherent.'}


In [69]:
df = pd.read_csv("/kaggle/working/reasoning.csv")
df.shape

(64625, 2)

In [70]:
dataset = load_dataset(
    "AI-MO/NuminaMath-CoT",
    split="train"
)

print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/train-00004-of-00005.parquet:   0%|          | 0.00/247M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/166k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/859494 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['source', 'problem', 'solution', 'messages'],
    num_rows: 859494
})


In [71]:
print(dataset[0]["problem"])
print("=" * 80)
print(dataset[0]["solution"])

Consider the terms of an arithmetic sequence: $-\frac{1}{3}, y+2, 4y, \ldots$. Solve for $y$.
For an arithmetic sequence, the difference between consecutive terms must be equal. Therefore, we can set up the following equations based on the sequence given:
\[ (y + 2) - \left(-\frac{1}{3}\right) = 4y - (y+2) \]

Simplify and solve these equations:
\[ y + 2 + \frac{1}{3} = 4y - y - 2 \]
\[ y + \frac{7}{3} = 3y - 2 \]
\[ \frac{7}{3} + 2 = 3y - y \]
\[ \frac{13}{3} = 2y \]
\[ y = \frac{13}{6} \]

Thus, the value of $y$ that satisfies the given arithmetic sequence is $\boxed{\frac{13}{6}}$.


In [73]:
from tqdm.auto import tqdm

PROMPT_TEMPLATES = [
    "Solve the following mathematics problem.\n\n{}",
    "Find the correct solution.\n\n{}",
    "Solve step by step.\n\n{}",
    "Determine the answer with proper reasoning.\n\n{}",
    "Work through the following problem.\n\n{}",
    "Answer the following mathematics question.\n\n{}",
    "Compute the result and justify each step.\n\n{}"
]

pairs = []

for sample in tqdm(
    dataset,
    total=len(dataset),
    desc="Processing NuminaMath"
):

    problem = sample["problem"].strip()
    solution = sample["solution"].strip()

    prompt = random.choice(PROMPT_TEMPLATES).format(problem)
    response = solution

    total_tokens = len(
        tokenizer.encode(prompt + "\n" + response).ids
    )

    if total_tokens <= MAX_TOKENS:
        pairs.append({
            "prompt": prompt,
            "response": response
        })

print(f"\nGenerated pairs: {len(pairs):,}")

print("\nExample Prompt:\n")
print(pairs[0]["prompt"])

print("\n" + "=" * 80 + "\n")

print("Example Response:\n")
print(pairs[0]["response"])

Processing NuminaMath:   0%|          | 0/859494 [00:00<?, ?it/s]


Generated pairs: 750,906

Example Prompt:

Compute the result and justify each step.

Consider the terms of an arithmetic sequence: $-\frac{1}{3}, y+2, 4y, \ldots$. Solve for $y$.


Example Response:

For an arithmetic sequence, the difference between consecutive terms must be equal. Therefore, we can set up the following equations based on the sequence given:
\[ (y + 2) - \left(-\frac{1}{3}\right) = 4y - (y+2) \]

Simplify and solve these equations:
\[ y + 2 + \frac{1}{3} = 4y - y - 2 \]
\[ y + \frac{7}{3} = 3y - 2 \]
\[ \frac{7}{3} + 2 = 3y - y \]
\[ \frac{13}{3} = 2y \]
\[ y = \frac{13}{6} \]

Thus, the value of $y$ that satisfies the given arithmetic sequence is $\boxed{\frac{13}{6}}$.


In [74]:
dataset.to_pandas()["source"].value_counts()

source
cn_k12            276554
synthetic_math    167874
orca_math         153314
olympiads         150563
synthetic_amc      62108
aops_forum         30192
math                7477
gsm8k               7342
amc_aime            4070
Name: count, dtype: int64

In [76]:
from tqdm.auto import tqdm

df = pd.DataFrame({
    "source": [x["source"] for x in tqdm(dataset, desc="Loading dataset")],
    "problem": [x["problem"] for x in dataset],
    "solution": [x["solution"] for x in dataset],
})

print("Before:", len(df))

df = df.drop_duplicates(subset=["problem"]).reset_index(drop=True)

print("After :", len(df))
print("Removed:", 859494 - len(df))

Loading dataset:   0%|          | 0/859494 [00:00<?, ?it/s]

Before: 859494
After : 816958
Removed: 42536


In [77]:
print(df["source"].value_counts())

source
cn_k12            268870
orca_math         149186
synthetic_math    148700
olympiads         143418
synthetic_amc      61916
aops_forum         30165
math                7351
amc_aime            3778
gsm8k               3574
Name: count, dtype: int64


In [78]:
from collections import defaultdict

stats = defaultdict(list)

for _, row in tqdm(df.iterrows(), total=len(df), desc="Measuring solution length"):
    stats[row["source"]].append(
        len(tokenizer.encode(row["solution"]).ids)
    )

for source in sorted(stats):
    lengths = stats[source]
    print(
        f"{source:16} "
        f"avg={sum(lengths)/len(lengths):7.1f}  "
        f"median={sorted(lengths)[len(lengths)//2]:4d}  "
        f"max={max(lengths):5d}"
    )

Measuring solution length:   0%|          | 0/816958 [00:00<?, ?it/s]

amc_aime         avg=  639.1  median= 615  max= 1535
aops_forum       avg=  986.7  median= 928  max= 4309
cn_k12           avg=  409.5  median= 347  max= 4895
gsm8k            avg=  266.8  median= 253  max=  744
math             avg=  462.9  median= 434  max= 1876
olympiads        avg=  937.4  median= 883  max= 5618
orca_math        avg=  271.9  median= 256  max= 1639
synthetic_amc    avg=  479.1  median= 465  max= 1435
synthetic_math   avg=  331.7  median= 301  max= 2761


In [79]:
TARGETS = {
    "orca_math": 20000,
    "synthetic_math": 10000,
    "cn_k12": 10000,
    "gsm8k": 3574,
    "math": 7351,
    "synthetic_amc": 5000,
    "amc_aime": 2500,
    "olympiads": 5000,
    "aops_forum": 2500,
}

balanced_df = (
    df.groupby("source", group_keys=False)
      .apply(lambda x: x.sample(min(len(x), TARGETS[x.name]), random_state=42))
      .reset_index(drop=True)
)

print(balanced_df["source"].value_counts())
print(f"\nTotal samples: {len(balanced_df):,}")

source
orca_math         20000
synthetic_math    10000
cn_k12            10000
math               7351
olympiads          5000
synthetic_amc      5000
gsm8k              3574
amc_aime           2500
aops_forum         2500
Name: count, dtype: int64

Total samples: 65,925


/tmp/ipykernel_58/3067631010.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), TARGETS[x.name]), random_state=42))


In [80]:
MAX_SOLUTION_TOKENS = 700

filtered_rows = []

for _, row in tqdm(
    balanced_df.iterrows(),
    total=len(balanced_df),
    desc="Filtering long solutions"
):
    if len(tokenizer.encode(row["solution"]).ids) <= MAX_SOLUTION_TOKENS:
        filtered_rows.append(row)

balanced_df = pd.DataFrame(filtered_rows).reset_index(drop=True)

print(balanced_df["source"].value_counts())
print(f"\nFinal samples: {len(balanced_df):,}")

Filtering long solutions:   0%|          | 0/65925 [00:00<?, ?it/s]

source
orca_math         19917
synthetic_math     9794
cn_k12             8718
math               6460
synthetic_amc      4656
gsm8k              3570
amc_aime           1584
olympiads          1399
aops_forum          588
Name: count, dtype: int64

Final samples: 56,686


In [82]:
from tqdm.auto import tqdm
import random

PROMPT_TEMPLATES = [
    "Solve the following mathematics problem.\n\n{}",
    "Solve this mathematics problem.\n\n{}",
    "Find the correct solution.\n\n{}",
    "Solve step by step.\n\n{}",
    "Determine the answer with proper reasoning.\n\n{}",
    "Work through the following problem.\n\n{}",
    "Answer the following mathematics question.\n\n{}",
    "Compute the result and justify each step.\n\n{}",
    "Find the value(s) that satisfy the problem.\n\n{}",
    "Show the complete mathematical solution.\n\n{}",
    "Carefully solve the following question.\n\n{}",
    "Find the answer using mathematical reasoning.\n\n{}",
    "Provide a complete solution.\n\n{}",
    "Work out the solution with clear reasoning.\n\n{}",
    "Derive the correct answer.\n\n{}",
    "Evaluate the following problem.\n\n{}",
    "Explain how to solve this problem.\n\n{}",
    "Use mathematics to solve the following question.\n\n{}",
    "Determine the correct result.\n\n{}",
    "Solve and explain your reasoning.\n\n{}",
]

pairs = []

for _, row in tqdm(
    balanced_df.iterrows(),
    total=len(balanced_df),
    desc="Creating prompt-response pairs"
):
    problem = str(row["problem"]).strip()
    solution = str(row["solution"]).strip()

    # Skip incomplete samples
    if not problem or not solution:
        continue

    prompt = random.choice(PROMPT_TEMPLATES).format(problem)

    total_tokens = len(
        tokenizer.encode(prompt + "\n" + solution).ids
    )

    if total_tokens <= MAX_TOKENS:
        pairs.append({
            "prompt": prompt,
            "response": solution,
            "source": row["source"]   # Keep for analysis
        })

random.shuffle(pairs)

print(f"Generated pairs: {len(pairs):,}")

print("\nSource distribution:\n")
print(
    pd.DataFrame(pairs)["source"].value_counts()
)

print("\nExample Prompt:\n")
print(pairs[0]["prompt"])

print("\n" + "=" * 100 + "\n")

print("Example Response:\n")
print(pairs[0]["response"])

Creating prompt-response pairs:   0%|          | 0/56686 [00:00<?, ?it/s]

Generated pairs: 56,556

Source distribution:

source
orca_math         19915
synthetic_math     9780
cn_k12             8708
math               6388
synthetic_amc      4649
gsm8k              3570
amc_aime           1573
olympiads          1399
aops_forum          574
Name: count, dtype: int64

Example Prompt:

Find the answer using mathematical reasoning.

In January the families visiting a national park see animals 26 times. In February the families that visit the national park see animals three times as many as were seen there in January. Then in March the animals are shyer and the families who visit the national park see animals half as many times as they were seen in February. How many times total did families see an animal in the first three months of the year?


Example Response:

To solve this problem, we follow the information given step by step:

1. In January, animals were seen **26 times**.
2. In February, animals were seen **three times** as many as in January. Therefore,

In [83]:
math_df = pd.DataFrame(pairs)

before = len(math_df)

math_df = math_df.drop_duplicates(
    subset=["prompt", "response"]
).reset_index(drop=True)

after = len(math_df)

print(f"Removed duplicates: {before - after:,}")
print(f"Math samples: {after:,}")

Removed duplicates: 0
Math samples: 56,556


In [84]:
reasoning_df = pd.read_csv("/kaggle/working/reasoning.csv")

print(f"Existing samples: {len(reasoning_df):,}")
reasoning_df.head()

Existing samples: 64,625


,prompt,response
0,Can you answer this? What is helping likely to...,The right answer is Good feelings.
1,What is the correct answer? Billy set of the c...,A suitable answer is Floor.
2,Answer the following question: You can find ca...,Usually Casino.
3,The man took paperwork to other people to cons...,The expected answer is Meeting.
4,Food on what kind of transport is normally fre...,Most commonly Space shuttle.


In [85]:
reasoning_df = pd.concat(
    [
        reasoning_df,
        math_df[["prompt", "response"]]
    ],
    ignore_index=True
)

print(f"Total samples after merge: {len(reasoning_df):,}")

Total samples after merge: 121,181


In [86]:
before = len(reasoning_df)

reasoning_df = (
    reasoning_df
    .drop_duplicates(subset=["prompt", "response"])
    .reset_index(drop=True)
)

after = len(reasoning_df)

print(f"Removed duplicates: {before - after:,}")
print(f"Final samples: {after:,}")

Removed duplicates: 0
Final samples: 121,181


In [87]:
reasoning_df = reasoning_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Dataset shuffled.")
print(reasoning_df.head())

Dataset shuffled.
                                              prompt  \
0  Fill in the blank using commonsense reasoning....   
1  Complete the sentence with the most appropriat...   
2  Work out the solution with clear reasoning.\n\...   
3  Generate an executable Python function generat...   
4  Use mathematics to solve the following questio...   

                                            response  
0                                           Benjamin  
1                                            fingers  
2  Let the income be 5x and the expenditure be 3x...  
3  Okay, let's tackle this problem. Hmm, the samp...  
4  First, we need to convert the speed of the man...  


In [88]:
OUTPUT_PATH = "/kaggle/working/reasoning.csv"

reasoning_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"✅ Saved {len(reasoning_df):,} samples")
print(f"Saved to: {OUTPUT_PATH}")

✅ Saved 121,181 samples
Saved to: /kaggle/working/reasoning.csv


In [89]:
from datasets import load_dataset

dataset = load_dataset(
    "open-r1/OpenR1-Math-220k",
    split="train"
)

print(dataset)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

data/train-00000-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/train-00001-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

data/train-00002-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

data/train-00007-of-00010.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

data/train-00008-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/train-00009-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/93733 [00:00<?, ? examples/s]

Dataset({
    features: ['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'],
    num_rows: 93733
})


In [90]:
print(dataset.column_names)

print("\nSample:\n")
print(dataset[0])

['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages']

Sample:

{'problem': '## Task B-1.3.\n\nA ship traveling along a river has covered $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream. For this journey, it took half an hour less than for traveling $30 \\mathrm{~km}$ upstream and $21 \\mathrm{~km}$ downstream, or half an hour more than for traveling $15 \\mathrm{~km}$ upstream and $42 \\mathrm{~km}$ downstream, assuming that both the ship and the river move uniformly.\n\nDetermine the speed of the ship in still water and the speed of the river.', 'solution': '## Solution.\n\nLet $t$ be the time required for the boat to travel $24 \\mathrm{~km}$ upstream and $28 \\mathrm{~km}$ downstream, $v_{R}$ the speed of the river, and $v_{B}$ the speed of the boat. When the boat is traveling upstream, its speed is $

In [91]:
from datasets import load_dataset

dataset = load_dataset(
    "open-thoughts/OpenThoughts-114k",
    split="train"
)

print(dataset)

Dataset({
    features: ['system', 'conversations'],
    num_rows: 113957
})


In [92]:
from pprint import pprint

print("System Prompt:\n")
print(dataset[0]["system"])

print("\nConversations:\n")
pprint(dataset[0]["conversations"])

System Prompt:

Your role as an assistant involves thoroughly exploring questions through a systematic long thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracing, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution. In the Thought section, detail your reasoning process using the specified format: <|begin_of_thought|> {thought with steps separated with '\n\n'} <|end_of_thought|> Each step should include detailed considerations such as analisying questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous steps. In the Solution section, based on various attempts, explorations, and reflections from the Thought section, systematically present the final solution that yo

In [93]:
from collections import Counter

Counter(len(x) for x in dataset["conversations"])

Counter({2: 113957})

In [94]:
import re
import pandas as pd
from tqdm.auto import tqdm

SOLUTION_RE = re.compile(
    r"<\|begin_of_solution\|>\s*(.*?)\s*<\|end_of_solution\|>",
    flags=re.DOTALL
)

rows = []

for sample in tqdm(dataset):
    prompt = sample["conversations"][0]["value"].strip()
    assistant = sample["conversations"][1]["value"]

    m = SOLUTION_RE.search(assistant)
    if not m:
        continue

    response = m.group(1).strip()

    if len(prompt) < 10 or len(response) < 10:
        continue

    rows.append({
        "prompt": prompt,
        "response": response
    })

df = pd.DataFrame(rows)

print("Extracted:", len(df))
display(df.head())

  0%|          | 0/113957 [00:00<?, ?it/s]

Extracted: 113946


,prompt,response
0,Generate an executable Python function generat...,"To solve this problem, we need to determine if..."
1,Generate an executable Python function generat...,"To solve this problem, we need to compute the ..."
2,Generate an executable Python function generat...,"To solve this problem, we need to determine th..."
3,Generate an executable Python function generat...,"To solve this problem, we need to determine if..."
4,Generate an executable Python function generat...,"To solve this problem, we need to determine if..."


In [97]:
from tqdm.auto import tqdm

MAX_TOKENS = 1024

filtered_rows = []

for row in tqdm(df.itertuples(index=False), total=len(df)):
    text = (
        "<bos>\n"
        + row.prompt
        + "\n\n"
        + row.response
        + "\n<eos>"
    )

    n_tokens = len(tokenizer.encode(text).ids)

    if n_tokens <= MAX_TOKENS:
        filtered_rows.append({
            "prompt": row.prompt,
            "response": row.response
        })

filtered_df = pd.DataFrame(filtered_rows)

print(f"Before: {len(df):,}")
print(f"After : {len(filtered_df):,}")
print(f"Removed: {len(df)-len(filtered_df):,}")

  0%|          | 0/113946 [00:00<?, ?it/s]

Before: 113,946
After : 87,628
Removed: 26,318


In [98]:
import random

random.seed(42)

for i in random.sample(range(len(filtered_df)), 100):
    print("=" * 100)
    print(filtered_df.iloc[i]["prompt"][:400])
    print()

"How can understanding the life cycle and transmission of zoonotic parasites contribute to the prevention and control of zoonotic diseases?"

Return your final response within \boxed{}. Given a cubic polynomial \( p(x) = a x^{3} + b x^{2} + c x + d \) with integer coefficients \( a, b, c, d \), and knowing that \( p(1) = 2015 \) and \( p(2) = 2017 \), prove that the equation \( p(x) = 2016 \) has no integer roots.

Generate an executable Python function generated from the given prompt. The function should take stdin as input and print the output. Simply call the function after the definition.We have N voting papers. The i-th vote (1 \leq i \leq N) has the string S_i written on it.
Print all strings that are written on the most number of votes, in lexicographical order.

-----Constraints-----
 - 1 \leq N \leq

Return your final response within \boxed{}. By a tropical polynomial we mean a function of the form
$$
p(x) = a_n \odot x^n \oplus a_{n-1} \odot x^{n-1} \oplus \cdots \oplus a_1 \

In [99]:
import pandas as pd

existing_df = pd.read_csv("/kaggle/working/reasoning.csv")

print("Existing :", len(existing_df))
print("OpenThoughts:", len(filtered_df))

merged_df = pd.concat(
    [existing_df, filtered_df],
    ignore_index=True
)

merged_df = (
    merged_df
    .drop_duplicates(subset=["prompt", "response"])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Final:", len(merged_df))

merged_df.to_csv(
    "/kaggle/working/reasoning.csv",
    index=False
)

print("Saved to /kaggle/working/reasoning.csv")

Existing : 121181
OpenThoughts: 87628
Final: 208809
Saved to /kaggle/working/reasoning.csv
